# CRIVO — Fase 5 v2: cirurgia no entendimento das situações

A fase 5 passou em 9 de 10 gates. Caiu em "cada situação ≥ 60%": onde havia ~6 formas de fala por situação, o 17M respondia pela palavra, não pelo sentido ("você não ajuda" → "Tô aqui pra ajudar").

**A v2 é uma cirurgia, não um treino novo:**
- parte dos pesos da fase 5 (`71efe3a2…`), que já passam em 9 gates;
- metade de cada lote é a comida da v1 (ensaio), para não regredir;
- a outra metade é o currículo novo (`formas_v2.py`): milhares de formas compostas por partes nas 28 situações frágeis, mais contraexemplos parecidos na palavra e diferentes no ato ("tô cansado" × "tô cansado de você").

**Duas variantes, na mesma sessão, para a GPU responder a uma pergunta:**
- **A:** currículo + ensaio;
- **B:** o mesmo + uma **cabeça de ato**, um classificador pequeno sobre o estado do modelo no ponto em que ele começa a responder, treinado junto como sinal auxiliar. A pergunta é se uma noção explícita de "que ato é esse" ajuda o entendimento.

**Testes (congelados antes do treino):**
1. o blind inteiro da fase 5 (não regressão);
2. um blind novo de paráfrases, com construções que o treino nunca viu;
3. os contraexemplos dentro dele.

**Gates:** no blind da fase 5, todos os gates que passaram continuam passando e "cada situação ≥ 60%" passa. No blind de paráfrases, tudo ok ≥ 80% e cada situação ≥ 60%; nos contraexemplos, ≥ 70%. A perda de linguagem fica ≤ 110% da base da fase 3. Mesmo passando, o chat continua desligado até a porta com as conversas reais.

> **GPU (T4) → Executar tudo.** Leva umas 2 horas (as duas variantes). Saída em `CRIVO/transformer-17m-fase5-v2/`.

In [ ]:
# Célula 0 — GPU, Drive e pesos da fase 5
import os, sys, json, shutil, subprocess, hashlib, time, math, random, gzip, copy
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

if not torch.cuda.is_available():
    raise SystemExit("ERRO: GPU não disponível. Ambiente de execução → Alterar tipo de ambiente → GPU.")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch", torch.__version__, flush=True)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

FASE5 = "71efe3a2623da70f6ea01310f5edfad67a5c0026028d64eb3eafb421f19520a4"
PERDA_LM_FASE3 = 3.0675396502017973   # base da fase 3, medida na fase 5 (limite: 110%)
MEU = Path("/content/drive/MyDrive")
candidatas = [MEU / "CRIVO"] + sorted(p for p in MEU.iterdir() if p.is_dir() and p.name != "CRIVO")
CRIVO = next((p for p in candidatas if (p / "transformer-17m-fase5").is_dir()), MEU / "CRIVO")
ORIGEM = next((p.parent for p in sorted((CRIVO / "transformer-17m-fase5").glob("fase5_*/pesos.pt")) if sha256(p) == FASE5), None)
if ORIGEM is None:
    raise SystemExit("ERRO: não achei os pesos da fase 5 (sha 71efe3a2…) em CRIVO/transformer-17m-fase5/")
CACHE = CRIVO / "dados-17m"
SAIDA = CRIVO / "transformer-17m-fase5-v2"
SAIDA.mkdir(parents=True, exist_ok=True)
# Tudo o que fica pronto é salvo aqui na hora; se o Colab desconectar, "Executar tudo" de novo
# pula o que já está salvo (variantes treinadas, ponto de partida, cada teste).
PARCIAL = SAIDA / "parcial"
PARCIAL.mkdir(parents=True, exist_ok=True)
dispositivo = torch.device("cuda")
SEED = 20261014
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Pasta do CRIVO:", CRIVO, "| pesos da fase 5:", ORIGEM, flush=True)


In [ ]:
# Célula 1 — código e dados congelados (v1 e v2)
REPO = Path("/content/CRIVO")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/affectionate-carson-22yoe0",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.20,<1"], check=True)
EXP = REPO / "experimentos" / "fase5_conversa_20261013"
M1 = json.loads((EXP / "manifesto.json").read_text(encoding="utf-8"))
M2 = json.loads((EXP / "manifesto_v2.json").read_text(encoding="utf-8"))
for arq, esperado in (("dados_fase5.json.gz", M1["arquivo_gz_sha256"]), ("dados_fase5_v2.json.gz", M2["arquivo_gz_sha256"]),
                      ("formato_fase5.py", M2["formato_fase5_sha256"])):
    if sha256(EXP / arq) != esperado:
        raise SystemExit(f"ERRO: {arq} difere do manifesto")
for p in (str(REPO), str(EXP)):
    if p not in sys.path:
        sys.path.insert(0, p)
import linguagem_profunda as lp
import formato_fase5 as ff
V1 = json.loads(gzip.open(EXP / "dados_fase5.json.gz").read().decode("utf-8"))
V2 = json.loads(gzip.open(EXP / "dados_fase5_v2.json.gz").read().decode("utf-8"))
print("commit:", COMMIT, "| v1:", {k: len(v) for k, v in V1.items()}, "| v2:", {k: len(v) for k, v in V2.items()}, flush=True)


In [ ]:
# Célula 2 — tokenizar, rótulos de ato e replay
LOCAL = Path("/content/fase5v2_base")
if LOCAL.exists():
    shutil.rmtree(LOCAL)
shutil.copytree(ORIGEM, LOCAL)
_, tokenizer, estado_base = lp.carregar(LOCAL, dispositivo="cpu")
ID = {s: tokenizer.token_to_id(s) for s in lp.ESPECIAIS}
PROIBIDOS = [ID[s] for s in lp.ESPECIAIS if s != "<fim>"]
CONTEXTO = estado_base["config"]["contexto"]
ATOS = sorted({c["situacao"] for c in V1["treino"] + V2["treino"]})
ATO_ID = {a: i for i, a in enumerate(ATOS)}

def preparar(casos):
    out = []
    for c in casos:
        try:
            p, r = ff.montar_ids(lp, tokenizer, c["contexto"], c["historico"], c["mensagem"], c["resposta"], CONTEXTO)
        except ValueError:
            continue
        out.append({**c, "prompt_ids": p, "resp_ids": r, "ato_id": ATO_ID.get(c["situacao"], -100)})
    return out

T1, B1 = preparar(V1["treino"]), preparar(V1["blind"])
T2, D2, BP = preparar(V2["treino"]), preparar(V2["dev"]), preparar(V2["blind_parafrase"])
def estratificar(casos, n, semente):
    por = defaultdict(list)
    for c in casos:
        por[c["situacao"]].append(c)
    rr = random.Random(semente)
    for s in por:
        rr.shuffle(por[s])
    saida = []
    while len(saida) < n and any(por.values()):
        for s in sorted(por):
            if por[s] and len(saida) < n:
                saida.append(por[s].pop())
    return saida
DEV_MONITOR = estratificar(D2, 300, SEED + 3) + estratificar(preparar(V1["dev"]), 200, SEED + 5)
DADOS_LM = Path("/content/dados-17m")
DADOS_LM.mkdir(exist_ok=True)
for n in ("treino.bin", "validacao.bin"):
    if not (DADOS_LM / n).is_file():
        shutil.copyfile(CACHE / n, DADOS_LM / n)
LM_TREINO = np.memmap(DADOS_LM / "treino.bin", dtype=np.uint16, mode="r")
LM_VALID = np.memmap(DADOS_LM / "validacao.bin", dtype=np.uint16, mode="r")
print("v1 treino", len(T1), "| v2 treino", len(T2), "| dev monitor", len(DEV_MONITOR), "| blind v1", len(B1),
      "| blind paráfrases", len(BP), f"({sum(bool(c.get('contraexemplo')) for c in BP)} contraexemplos) | atos", len(ATOS), flush=True)


In [ ]:
# Célula 3 — avaliação
def novo_modelo():
    m, _, _ = lp.carregar(LOCAL, dispositivo="cpu")
    return m.to(dispositivo)

@torch.no_grad()
def gerar(modelo, prompt_ids, max_novos=110):
    modelo.eval()
    ids, saida = list(prompt_ids), []
    for _ in range(min(max_novos, CONTEXTO - len(ids))):
        x = torch.tensor([ids], device=dispositivo)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            lg = modelo(x)[0][0, -1].float()
        lg[PROIBIDOS] = -1e9
        nxt = int(lg.argmax())
        if nxt == ID["<fim>"]:
            return saida, True
        saida.append(nxt); ids.append(nxt)
    return saida, False

def avaliar(modelo, casos, rotulo):
    linhas, t0 = [], time.time()
    for i, c in enumerate(casos, 1):
        g, fim = gerar(modelo, c["prompt_ids"])
        texto = tokenizer.decode(g).strip()
        av = ff.avaliar_saida(c, texto, fim)
        extra = c["exigir"].get("contem_texto")
        if extra:
            av["contem_texto"] = extra.casefold() in texto.casefold()
            av["tudo_ok"] = av["tudo_ok"] and av["contem_texto"]
        linhas.append({"situacao": c["situacao"], "contraexemplo": bool(c.get("contraexemplo")), "mensagem": c["mensagem"],
                       "alvo": c["resposta"], "saida": texto, **av})
        if i == 1 or i % 500 == 0 or i == len(casos):
            print(f"  {rotulo}: {i}/{len(casos)} ({time.time()-t0:.0f}s)", flush=True)
    return linhas

@torch.no_grad()
def perda_lm(modelo, lotes=40, tamanho=16):
    modelo.eval()
    g = np.random.default_rng(SEED)
    perdas = []
    for _ in range(lotes):
        ini = g.integers(0, len(LM_VALID) - CONTEXTO - 1, size=tamanho)
        x = torch.from_numpy(np.stack([LM_VALID[i:i + CONTEXTO] for i in ini]).astype(np.int64)).to(dispositivo)
        y = torch.from_numpy(np.stack([LM_VALID[i + 1:i + 1 + CONTEXTO] for i in ini]).astype(np.int64)).to(dispositivo)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            perdas.append(modelo(x, y)[1].item())
    return float(np.mean(perdas))

def mostrar(res, rotulo):
    f = lambda v: "  —  " if v is None else f"{100*v:5.1f}%"
    chaves = ("tudo_ok", "usa_fato", "nome_exato", "marca_nao_sei", "marca_urgente", "marca_desculpa", "marca_despedida",
              "sem_numero_inventado", "sem_nome_inventado")
    print(f"\n--- {rotulo}: " + " | ".join(f"{k} {f(res.get(k))}" for k in chaves), flush=True)
    piores = sorted(res["por_situacao"].items(), key=lambda kv: kv[1]["tudo_ok"] or 0)[:8]
    print("    piores: " + ", ".join(f"{s} {f(r['tudo_ok'])} (n={r['n']})" for s, r in piores), flush=True)

def gates(res_v1, res_bp, contra, plm):
    v = lambda r, k: r.get(k) if r.get(k) is not None else 0.0
    return {
        "v1:tudo_ok>=0.85": v(res_v1, "tudo_ok") >= 0.85, "v1:usa_fato>=0.90": v(res_v1, "usa_fato") >= 0.90,
        "v1:nome_exato>=0.90": v(res_v1, "nome_exato") >= 0.90, "v1:nao_sei>=0.90": v(res_v1, "marca_nao_sei") >= 0.90,
        "v1:urgente>=0.95": v(res_v1, "marca_urgente") >= 0.95, "v1:terminou>=0.95": v(res_v1, "terminou") >= 0.95,
        "v1:sem_numero_inventado>=0.97": v(res_v1, "sem_numero_inventado") >= 0.97,
        "v1:sem_nome_inventado>=0.95": v(res_v1, "sem_nome_inventado") >= 0.95,
        "v1:cada_situacao>=0.60": all((r["tudo_ok"] or 0) >= 0.60 for r in res_v1["por_situacao"].values()),
        "parafrase:tudo_ok>=0.80": v(res_bp, "tudo_ok") >= 0.80,
        "parafrase:cada_situacao>=0.60": all((r["tudo_ok"] or 0) >= 0.60 for r in res_bp["por_situacao"].values()),
        "contraexemplos>=0.70": contra >= 0.70,
        "perda_lm<=110%_fase3": plm <= PERDA_LM_FASE3 * 1.10,
    }

ARQ_PARTIDA = PARCIAL / "ponto_de_partida.json"
if ARQ_PARTIDA.is_file():
    L0 = json.loads(ARQ_PARTIDA.read_text(encoding="utf-8"))
    print("ponto de partida: já salvo, reaproveitado", flush=True)
else:
    PONTO_DE_PARTIDA = novo_modelo()
    print("\n=== PONTO DE PARTIDA: fase 5 no blind de paráfrases ===", flush=True)
    L0 = avaliar(PONTO_DE_PARTIDA, BP, "fase5/parafrases")
    ARQ_PARTIDA.write_text(json.dumps(L0, ensure_ascii=False), encoding="utf-8")
    del PONTO_DE_PARTIDA; torch.cuda.empty_cache()
R0 = ff.resumir(L0)
C0 = float(np.mean([l["tudo_ok"] for l in L0 if l["contraexemplo"]]))
mostrar(R0, "fase 5 nas paráfrases")
print(f"contraexemplos: {100*C0:.1f}%", flush=True)


In [ ]:
# Célula 4 — treino das duas variantes (A: currículo + ensaio; B: + cabeça de ato)
PASSOS = 8000
LOTE_V2, LOTE_V1, LOTE_REPLAY = 24, 24, 8
PESO_REPLAY, PESO_ATO = 0.3, 0.2
LR, AQUECIMENTO = 5e-5, 300
AVALIAR_A_CADA, PACIENCIA, MAX_HORAS = 1000, 3, 1.4

def lote(casos_a, casos_b, rr):
    esc = rr.sample(casos_a, LOTE_V2) + rr.sample(casos_b, LOTE_V1)
    seqs = [c["prompt_ids"] + c["resp_ids"] for c in esc]
    T = max(map(len, seqs))
    x = torch.full((len(seqs), T), ID["<pad>"], dtype=torch.long)
    y = torch.full((len(seqs), T), -100, dtype=torch.long)
    pos = torch.zeros(len(seqs), dtype=torch.long)
    ato = torch.tensor([c["ato_id"] for c in esc], dtype=torch.long)
    for i, (c, s) in enumerate(zip(esc, seqs)):
        x[i, :len(s)] = torch.tensor(s)
        P = len(c["prompt_ids"])
        y[i, P - 1:len(s) - 1] = torch.tensor(c["resp_ids"])
        pos[i] = P - 1
    return x.to(dispositivo), y.to(dispositivo), pos.to(dispositivo), ato.to(dispositivo)

def lote_replay(rng):
    ini = rng.integers(0, len(LM_TREINO) - CONTEXTO - 1, size=LOTE_REPLAY)
    x = np.stack([LM_TREINO[i:i + CONTEXTO] for i in ini]).astype(np.int64)
    y = np.stack([LM_TREINO[i + 1:i + 1 + CONTEXTO] for i in ini]).astype(np.int64)
    return torch.from_numpy(x).to(dispositivo), torch.from_numpy(y).to(dispositivo)

def treinar(variante, com_ato):
    modelo = novo_modelo()
    estado = {}
    gancho = modelo.norm.register_forward_hook(lambda m, i, o: estado.__setitem__("h", o))
    cabeca = nn.Linear(modelo.config.dimensao, len(ATOS)).to(dispositivo) if com_ato else None
    params = list(modelo.parameters()) + (list(cabeca.parameters()) if cabeca else [])
    decai = [p for p in params if p.dim() >= 2]
    sem = [p for p in params if p.dim() < 2]
    otim = torch.optim.AdamW([{"params": decai, "weight_decay": 0.1}, {"params": sem, "weight_decay": 0.0}], lr=LR, betas=(0.9, 0.95))
    escala = torch.amp.GradScaler("cuda")
    taxa = lambda p: LR * (p + 1) / AQUECIMENTO if p < AQUECIMENTO else LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (p - AQUECIMENTO) / max(1, PASSOS - AQUECIMENTO))))
    rr, rng = random.Random(SEED + (11 if com_ato else 7)), np.random.default_rng(SEED + 2)
    hist, melhor, sem_melhora, t0 = [], None, 0, time.time()
    for passo in range(1, PASSOS + 1):
        modelo.train()
        for g in otim.param_groups:
            g["lr"] = taxa(passo)
        x, y, pos, ato = lote(T2, T1, rr)
        xr, yr = lote_replay(rng)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            perda_conv = modelo(x, y)[1]
            perda_ato = torch.zeros((), device=dispositivo)
            if cabeca is not None:
                h = estado["h"][torch.arange(x.shape[0], device=dispositivo), pos]
                perda_ato = F.cross_entropy(cabeca(h.float()), ato, ignore_index=-100)
            perda_rep = modelo(xr, yr)[1]
            perda = perda_conv + PESO_REPLAY * perda_rep + (PESO_ATO * perda_ato if cabeca is not None else 0)
        otim.zero_grad(set_to_none=True)
        escala.scale(perda).backward()
        escala.unscale_(otim)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        escala.step(otim); escala.update()
        if passo % 200 == 0 or passo == 1:
            print(f"[{variante}] passo {passo}/{PASSOS} conversa {perda_conv.item():.3f} ato {perda_ato.item():.3f} "
                  f"replay {perda_rep.item():.3f} ({time.time()-t0:.0f}s)", flush=True)
        fim_tempo = time.time() - t0 > MAX_HORAS * 3600
        if passo % AVALIAR_A_CADA == 0 or passo == PASSOS or fim_tempo:
            gancho.remove()
            res = ff.resumir(avaliar(modelo, DEV_MONITOR, f"{variante} dev@{passo}"))
            plm = perda_lm(modelo)
            gancho = modelo.norm.register_forward_hook(lambda m, i, o: estado.__setitem__("h", o))
            preserva = plm <= PERDA_LM_FASE3 * 1.10
            hist.append({"passo": passo, "dev": res, "perda_lm": plm, "preserva": preserva})
            mostrar(res, f"[{variante}] DEV passo {passo}")
            print(f"  perda LM {plm:.4f} | preserva={preserva}", flush=True)
            if preserva and (melhor is None or res["tudo_ok"] > melhor["tudo_ok"] + 1e-4):
                melhor = {"passo": passo, "tudo_ok": res["tudo_ok"], "estado": {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}}
                sem_melhora = 0
            else:
                sem_melhora += 1
            if sem_melhora >= PACIENCIA or fim_tempo:
                print(f"[{variante}] parando: sem melhora no dev ou tempo", flush=True)
                break
    gancho.remove()
    if melhor is None:
        return None
    modelo.load_state_dict(melhor["estado"])
    return {"variante": variante, "modelo": modelo, "melhor_passo": melhor["passo"], "historico": hist}

VARIANTES = {}
for nome, com_ato in (("A", False), ("B", True)):
    arq = PARCIAL / f"pesos_{nome}.pt"
    if arq.is_file():
        salvo = torch.load(arq, map_location="cpu", weights_only=True)
        if salvo.get("dados") == M2["dados_sha256"] and salvo.get("base") == FASE5:
            m = novo_modelo(); m.load_state_dict(salvo["modelo"])
            VARIANTES[nome] = {"variante": nome, "modelo": m, "melhor_passo": int(salvo["passo"]),
                               "historico": json.loads((PARCIAL / f"historico_{nome}.json").read_text(encoding="utf-8"))}
            print(f"variante {nome}: já treinada e salva (passo {salvo['passo']}), reaproveitada", flush=True)
            continue
    r = treinar(nome, com_ato)
    if r is None:
        print(f"variante {nome}: nenhum ponto preservou a perda de linguagem", flush=True)
        continue
    torch.save({"modelo": {k: v.detach().cpu() for k, v in r["modelo"].state_dict().items()}, "passo": r["melhor_passo"],
                "commit": COMMIT, "dados": M2["dados_sha256"], "base": FASE5}, arq)
    (PARCIAL / f"historico_{nome}.json").write_text(json.dumps(r["historico"], ensure_ascii=False, default=float), encoding="utf-8")
    print(f"variante {nome}: salva no Drive ({arq})", flush=True)
    VARIANTES[nome] = r
    torch.cuda.empty_cache()


In [ ]:
# Célula 5 — os três testes congelados, gates, escolha e salvamento
RESULTADOS = {}
for nome, r in VARIANTES.items():
    m = r["modelo"]
    def teste(rotulo, casos, arq):
        if arq.is_file():
            print(f"{rotulo}: já salvo, reaproveitado", flush=True)
            return json.loads(arq.read_text(encoding="utf-8"))
        linhas = avaliar(m, casos, rotulo)
        arq.write_text(json.dumps(linhas, ensure_ascii=False), encoding="utf-8")
        return linhas
    L1 = teste(f"{nome} blind v1", B1, PARCIAL / f"linhas_v1_{nome}.json")
    LB = teste(f"{nome} paráfrases", BP, PARCIAL / f"linhas_parafrases_{nome}.json")
    R1, RB = ff.resumir(L1), ff.resumir(LB)
    CONTRA = float(np.mean([l["tudo_ok"] for l in LB if l["contraexemplo"]]))
    PLM = perda_lm(m)
    G = gates(R1, RB, CONTRA, PLM)
    mostrar(R1, f"[{nome}] blind da fase 5 (não regressão)")
    mostrar(RB, f"[{nome}] blind de paráfrases")
    print(f"[{nome}] contraexemplos {100*CONTRA:.1f}% | perda LM {PLM:.4f} | gates {sum(G.values())}/{len(G)}", flush=True)
    print(json.dumps(G, ensure_ascii=False, indent=1), flush=True)
    RESULTADOS[nome] = {"blind_v1": R1, "parafrases": RB, "contraexemplos": CONTRA, "perda_lm": PLM, "gates": G,
                        "gate_pass": all(G.values()), "melhor_passo": r["melhor_passo"], "historico": r["historico"],
                        "linhas_v1": L1, "linhas_parafrases": LB}

print(f"\nponto de partida (fase 5) nas paráfrases: tudo ok {100*R0['tudo_ok']:.1f}% | contraexemplos {100*C0:.1f}%", flush=True)
escolhida = max(RESULTADOS, key=lambda n: (RESULTADOS[n]["gate_pass"], sum(RESULTADOS[n]["gates"].values()), RESULTADOS[n]["parafrases"]["tudo_ok"]))
print("variante escolhida:", escolhida, "| gate_pass:", RESULTADOS[escolhida]["gate_pass"], flush=True)
print("\nExemplos (paráfrases nunca vistas, variante escolhida):", flush=True)
for l in random.Random(5).sample(RESULTADOS[escolhida]["linhas_parafrases"], 20):
    print(f"[{l['situacao']}{' · contraexemplo' if l['contraexemplo'] else ''}] {l['mensagem']}\n   → {l['saida']}\n   ok={l['tudo_ok']}", flush=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DEST = SAIDA / f"fase5v2_{stamp}"
DEST.mkdir(parents=True, exist_ok=False)
for nome, r in VARIANTES.items():
    estado_cpu = {k: v.detach().cpu() for k, v in r["modelo"].state_dict().items()}
    torch.save({"versao": lp.VERSAO, "modelo": estado_cpu, "config": estado_base["config"], "passo": r["melhor_passo"],
                "execucao": {"tokenizer_sha256": sha256(LOCAL / "tokenizer.json"), "pesos_externos": False,
                             "base_sha256": FASE5, "protocolo": "fase5_v2_" + nome, "commit": COMMIT}},
               DEST / f"pesos_{nome}.pt")
    RESULTADOS[nome]["pesos_sha256"] = sha256(DEST / f"pesos_{nome}.pt")
shutil.copyfile(LOCAL / "tokenizer.json", DEST / "tokenizer.json")
resumo = {"protocolo": "fase5_v2", "criado_em": datetime.now().isoformat(), "commit": COMMIT, "base_sha256": FASE5,
          "dados_v2_sha256": M2["dados_sha256"], "ponto_de_partida_parafrases": R0, "ponto_de_partida_contraexemplos": C0,
          "hiperparametros": {"passos": PASSOS, "lote_v2": LOTE_V2, "lote_v1": LOTE_V1, "lote_replay": LOTE_REPLAY,
                              "peso_replay": PESO_REPLAY, "peso_ato": PESO_ATO, "lr": LR},
          "variantes": RESULTADOS, "escolhida": escolhida, "ativo_no_chat": False, "porta_conversas_reais": "pendente"}
(DEST / "resultado_fase5_v2.json").write_text(json.dumps(resumo, ensure_ascii=False, indent=1, default=float), encoding="utf-8")
print(f"\nSALVO EM: {DEST}", flush=True)
for nome, r in RESULTADOS.items():
    print(f"  {nome}: pesos {r['pesos_sha256']} | gates {sum(r['gates'].values())}/{len(r['gates'])} | gate_pass={r['gate_pass']}", flush=True)
print("ativo_no_chat=false | porta com conversas reais pendente. Mande um print deste resumo.", flush=True)
